In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

## Deduplication of Images

### Classes labeled Violation vs NoViolation

In [2]:
def reduce_to_violation(text: str):
    if "NoHelmet" in text:
        return "1" # Violation
    else:
        return "0" # No Violation

In [3]:
input_path = '/kaggle/input/helmet-detection/'

train_df = pd.read_csv(f"{input_path}helmet_detection_train.csv")
test_df = pd.read_csv(f"{input_path}helmet_detection_test.csv")
val_df = pd.read_csv(f"{input_path}helmet_detection_validation.csv")

train_violation_classes = train_df.copy()
train_violation_classes['label'] = train_violation_classes['label'].apply(reduce_to_violation)

val_violation_classes = val_df.copy()
val_violation_classes['label'] = val_violation_classes['label'].apply(reduce_to_violation)

test_violation_classes = test_df.copy()
test_violation_classes['label'] = test_violation_classes['label'].apply(reduce_to_violation)

In [4]:
def remove_redundant_images(df):
    '''
    Removes redundant images. Make sure "image_only" column exists in df (not present in original df), 
    which should be the highway key without the image number.
    
    Args:
        df: Any train, test, val df to remove the redundant images, thus alleviating training constraints

    Returns:
        df: The df without redundant images returned
    '''
    df_columns = df.columns
    
    # Check for required columns 
    df_required_columns = ['image_name', 'label', 'track_id', 'x1', 'x2', 'y1', 'y2', 'image_only']
    required_columns_missing = any(required_column not in df_columns for required_column in df_required_columns)

    if required_columns_missing:
        raise KeyError("Missing required columns from df. They are 'image_name', 'label', 'track_id', 'x1', 'x2', 'y1', 'y2', 'image_only'")
    
    # label + track_id signature
    df['obj_signature'] = (
        df['label'].astype(str) + ":" + 
        df['track_id'].astype(str)
    )
    
    # A unique scene_signature for every image name
    # Sorting is crucial so that [A:Violation, B:NoViolation] matches [B:NoViolation, A:Violation]
    scene_signatures = df.groupby('image_name')['obj_signature'].apply(
        lambda x: "|".join(sorted(x))
    ).reset_index()
    
    # Bringing back the 'image_only' context for deduplication
    scene_signatures = scene_signatures.merge(
        df[['image_name', 'image_only']].drop_duplicates(),
        on='image_name'
    )
    
    # 4. Droping duplicates on the Scene Signature and the highway (image_only)
    # This keeps the first image that showed this specific configuration of objects/labels
    unique_image_names = scene_signatures.drop_duplicates(
        subset=['image_only', 'obj_signature'], 
        keep='first'
    )['image_name']
    
    # Filter original dataframe using unique image names
    df_train_final = df[df['image_name'].isin(unique_image_names)].copy()
    
    # Removing the helper column from the final result
    df_train_final.drop(columns=['obj_signature'], inplace=True)

    return df_train_final

In [5]:
train_violation_classes["image_only"] = train_violation_classes["image_name"]
train_violation_classes["image_only"] = train_violation_classes["image_only"].apply(lambda image: "_".join(image.split("_")[:-1]))
df_train_final = remove_redundant_images(train_violation_classes)

# 'image_name' is named 'image-id' in val
val_violation_classes["image_name"] = val_violation_classes["image_id"]
val_violation_classes.drop('image_id', axis=1, inplace=True)
val_violation_classes["image_only"] = val_violation_classes["image_name"]
val_violation_classes["image_only"] = val_violation_classes["image_only"].apply(lambda image: "_".join(image.split("_")[:-1]))
df_val_final = remove_redundant_images(val_violation_classes)

test_violation_classes["image_only"] = test_violation_classes["image_name"]
test_violation_classes["image_only"] = test_violation_classes["image_only"].apply(lambda image: "_".join(image.split("_")[:-1]))
df_test_final = remove_redundant_images(test_violation_classes)


In [6]:
train_violation_classes['label'].value_counts()

label
0    121107
1     78496
Name: count, dtype: int64

In [7]:
df_train_final['label'].value_counts()

label
0    29171
1    14857
Name: count, dtype: int64

In [8]:
df_train_final.head()

,image_name,label,track_id,x1,y1,x2,y2,image_only
0,Bago_highway_11_01,1,_msurfy1df,1481,605,1549,714,Bago_highway_11
10,Bago_highway_11_100,0,_kj96uh9tz,831,637,927,826,Bago_highway_11
16,Bago_highway_11_16,0,_kj96uh9tz,943,626,1050,796,Bago_highway_11
17,Bago_highway_11_16,1,_msurfy1df,1388,596,1453,706,Bago_highway_11
36,Bago_highway_11_26,1,_n0a1jjoqv,1815,569,1920,696,Bago_highway_11


## ETL

In [9]:
from pathlib import Path
import shutil

# Output Directories
dataset_dir = Path('/kaggle/working/helmet-dataset-yolo-compatible')
dataset_dir.mkdir(exist_ok=True)

train_dir = Path('/kaggle/working/helmet-dataset-yolo-compatible/train')
train_dir.mkdir(exist_ok=True)
train_images_dir = train_dir / "images"
train_images_dir.mkdir(parents=True, exist_ok=True)
train_labels_dir = train_dir / "labels"
train_labels_dir.mkdir(parents=True, exist_ok=True)

val_dir = Path('/kaggle/working/helmet-dataset-yolo-compatible/val')
val_dir.mkdir(exist_ok=True)
val_images_dir = val_dir / "images"
val_images_dir.mkdir(parents=True, exist_ok=True)
val_labels_dir = val_dir / "labels"
val_labels_dir.mkdir(parents=True, exist_ok=True)

test_dir = Path('/kaggle/working/helmet-dataset-yolo-compatible/test')
test_dir.mkdir(exist_ok=True)
test_images_dir = test_dir / "images"
test_images_dir.mkdir(parents=True, exist_ok=True)
test_labels_dir = test_dir / "labels"
test_labels_dir.mkdir(parents=True, exist_ok=True)

# Input Directories
source_train = Path('/kaggle/input/helmet-detection/training_v2')
source_val = Path('/kaggle/input/helmet-detection/validation_v2')
source_test = Path('/kaggle/input/helmet-detection/test_v2')

In [10]:
test_read = source_train / "Bago_highway_11/annotations/18.txt"
test = test_read.read_text(encoding='utf-8')
test.split("\n")

['DHelmet _kj96uh9tz 0.5208333333333334 0.6601851851851852 0.053125 0.15925925925925927',
 'DNoHelmet _msurfy1df 0.7346354166666667 0.6 0.034895833333333334 0.09814814814814815',
 '']

In [11]:
# Helper Function to Transform labels Text to Yolo Format and save to destination
def _transform_labels_file(source_file, destination_dir):
    content = source_file.read_text(encoding='utf-8')

    # Separate objects
    objects_to_detect = [
        object_to_detect.strip()
        for object_to_detect in content.split('\n') 
        if object_to_detect]

    cleaned_content = list()

    # tranform to yolo compatible
    for object_to_detect in objects_to_detect:
        texts = object_to_detect.split(" ")
        cleaned_content.append(' '.join([reduce_to_violation(texts[0])] + texts[2:]))

    yolo_ready_text = "\n".join(cleaned_content)

    # Pick the highway name 
    file_name = source_file.parent.parent.name + '_' + source_file.name
    file_path = destination_dir / file_name

    file_path.write_text(yolo_ready_text, encoding='utf-8')

def _handle_image_file(source_file, destination_dir):
    file_name = source_file.parent.parent.name + '_' + source_file.name
    file_path = destination_dir / file_name

    shutil.copy2(source_file, file_path)

# Function to handle each split 
def etl_split(source, destination_images, destination_labels, deduplicated_df):
    deduplicated_df_columns = deduplicated_df.columns
    
    # Check for required columns 
    deduplicated_df_required_columns = ['image_name', 'label', 'track_id', 'x1', 'x2', 'y1', 'y2', 'image_only']
    required_columns_missing = any(required_column not in deduplicated_df_columns for required_column in deduplicated_df_required_columns)

    if required_columns_missing:
        raise KeyError("Missing required columns from deduplicated_df. They are 'image_name', 'label', 'track_id', 'x1', 'x2', 'y1', 'y2', 'image_only'")

    print(f"Processing {destination_images.parent.name} Set\n")

    # Highway sanity check
    highways = set(deduplicated_df['image_only'].unique())

    # Images set to keep
    images_set = set(deduplicated_df['image_name'].unique()) 

    for highway_folder in source.iterdir():
        if highway_folder.is_dir():
            highway_name = highway_folder.name
            if highway_name not in highways:
                print(f"{highway_name} not in deduplicated list. Skipping...")
                continue 
                
            img_dir = highway_folder / "images"
            ann_dir = highway_folder / "annotations"
            
            images = sorted(list(img_dir.glob('*.jpg'))) 
            annotations = sorted(list(ann_dir.glob('*.txt')))
    
            # Warn mismatch
            if len(images) != len(annotations):
                print(f"Warning: Mismatch in {highway_name}!")
                print(f"Images: {len(images)} | Annotations: {len(annotations)}")
    
            for img_path, ann_path in zip(images, annotations):
                image_full_name = highway_name + '_' + img_path.stem

                if image_full_name not in images_set:
                    continue
                try:
                    _transform_labels_file(ann_path, destination_labels)
                    _handle_image_file(img_path, destination_images)
                except Exception as e:
                    print(f"Error during transformation and load: {e}")

    print(f"ETL completed for {destination_images.parent.name} Set.")

In [12]:
etl_split(source_train, train_images_dir, train_labels_dir, df_train_final)

Processing train Set

ETL completed for train Set.


In [13]:
etl_split(source_test, test_images_dir, test_labels_dir, df_test_final)

Processing test Set

ETL completed for test Set.


In [14]:
etl_split(source_val, val_images_dir, val_labels_dir, df_val_final)

Processing val Set

ETL completed for val Set.


In [15]:
# dataframes dir to store original and deduplicated
dataframes_dir = dataset_dir / 'dataframes'
dataframes_dir.mkdir(exist_ok=True)

dataframes_dir_str = str(dataframes_dir.resolve())

train_original_filename = 'helmet_detection_train.csv'
test_original_filename = 'helmet_detection_test.csv'
validation_original_filename = 'helmet_detection_validation.csv'

train_original_path = Path(input_path) / train_original_filename
test_original_path = Path(input_path) / test_original_filename
validation_original_path = Path(input_path) / validation_original_filename

# Copy Original datasets
shutil.copy2(train_original_path, dataframes_dir / train_original_filename)
shutil.copy2(test_original_path, dataframes_dir / test_original_filename)
shutil.copy2(validation_original_path, dataframes_dir / validation_original_filename)

# Save deduplicated datasets as well
df_train_final.to_csv(f'{dataframes_dir_str}/helmet_detection_train_deduplicated.csv', index=False)
df_test_final.to_csv(f'{dataframes_dir}/helmet_detection_test_deduplicated.csv', index=False)
df_val_final.to_csv(f'{dataframes_dir}/helmet_detection_validation_deduplicated.csv', index=False)

In [16]:
import yaml

# Define the dataset structure
dataset_root = "/kaggle/working/helmet-dataset-yolo-compatible/"

yaml_data = {
    'path': ".",    
    'train': 'train/images', 
    'val': 'val/images',    
    'test': 'test/images', 
    
    'nc': 2,                 
    'names': [
        'Helmet',            
        'NoHelmet'          
    ]
}

# Write the yaml file
yaml_file_path = Path(dataset_root) / "data.yaml"

with open(yaml_file_path, 'w') as f:
    yaml.dump(yaml_data, f, default_flow_style=False)

print(f"data.yaml successfully created at: {yaml_file_path}")

data.yaml successfully created at: /kaggle/working/helmet-dataset-yolo-compatible/data.yaml
